# Language Identification for 23 Indian Languages

TF-IDF (word 1-2 grams + char 2-4 grams) + softmax logistic regression, evaluated with macro-F1.
All three are implemented from scratch. `sklearn` is not used anywhere.

## Settings

In [1]:
import os, re, unicodedata, requests
import numpy as np
import pandas as pd
from collections import Counter
from scipy import sparse

SENTENCES_PER_LANGUAGE = 1000
WORD_NGRAM_SIZES = (1, 2)
CHAR_NGRAM_SIZES = (2, 3, 4)
MAX_FEATURES     = 500_000

EPOCHS        = 40
LEARNING_RATE = 5.0
BATCH_SIZE    = 128

SEED = 42
DATA_FILE = os.path.join("data", "langid_dataset.csv")
os.makedirs("data", exist_ok=True)

c:\Users\KURUV PATEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


## 1. Dataset

In [2]:
LANGUAGES = {
    "as": "Assamese", "bd": "Bodo",      "bn": "Bengali",   "dg": "Dogri",
    "gom": "Konkani", "gu": "Gujarati",  "hi": "Hindi",     "kha": "Khasi",
    "kn": "Kannada",  "ks": "Kashmiri",  "mai": "Maithili", "ml": "Malayalam",
    "mni": "Manipuri","mr": "Marathi",   "ne": "Nepali",    "or": "Odia",
    "pa": "Punjabi",  "sa": "Sanskrit",  "sat": "Santali",  "sd": "Sindhi",
    "ta": "Tamil",    "te": "Telugu",    "ur": "Urdu",
}

LABELS = sorted(LANGUAGES)                             # label 0 = "as", label 1 = "bd", ...
label_to_number = {code: i for i, code in enumerate(LABELS)}

In [3]:
# Sentence terminators: danda, double danda, Arabic full stop, Ol Chiki, ASCII.
split_on_endings = re.compile("[।॥۔᱾᱿.?!]+")


def split_into_sentences(paragraph):
    sentences = []
    for piece in split_on_endings.split(paragraph):
        piece = " ".join(piece.split())

        # Unicode note: 'ा'.isalpha() is False (it is a combining mark, category Mc),
        # so we accept categories L* and M* instead of using isalpha().
        letters = sum(unicodedata.category(c)[0] in ("L", "M") for c in piece)

        if 25 <= len(piece) <= 300 and len(piece.split()) >= 4 and letters >= 0.5 * len(piece):
            sentences.append(piece)
    return sentences


split_into_sentences("भारत एक विशाल और विविधतापूर्ण देश है। 12345 6789 000.")

['भारत एक विशाल और विविधतापूर्ण देश है']

In [4]:
# The corpus files are many GB, so we stream them and stop once we have enough sentences.
URL = "https://huggingface.co/datasets/ai4bharat/IndicCorpV2/resolve/main/data/{}.txt"


def download_sentences(code, how_many):
    sentences, seen = [], set()
    remote_name = "hi-1" if code == "hi" else code      # Hindi is split into 3 files

    with requests.get(URL.format(remote_name), stream=True, timeout=120) as response:
        response.raise_for_status()
        response.encoding = "utf-8"

        for paragraph in response.iter_lines(decode_unicode=True):
            if paragraph:
                for sentence in split_into_sentences(paragraph)[:3]:   # <=3 per paragraph
                    if sentence not in seen:
                        seen.add(sentence)
                        sentences.append(sentence)
            if len(sentences) >= how_many:
                break                                   # closes the connection early

    return sentences[:how_many]

In [5]:
if os.path.exists(DATA_FILE):
    dataset = pd.read_csv(DATA_FILE)
else:
    rows = []
    for code in LABELS:
        found = download_sentences(code, SENTENCES_PER_LANGUAGE)
        print(f"{code:<4} {len(found)} sentences")
        rows += [{"label": code, "text": t} for t in found]
    dataset = pd.DataFrame(rows)
    dataset.to_csv(DATA_FILE, index=False)

print(f"{len(dataset)} sentences, {dataset.groupby('label').size().iloc[0]} per language")

23000 sentences, 1000 per language


## 2. Stratified 80 / 10 / 10 split

In [ ]:
def stratified_split(table):
    rng = np.random.default_rng(SEED)
    train, validation, test = [], [], []

    for label, group in table.groupby("label"):         # split inside each language
        group = group.iloc[rng.permutation(len(group))]
        n_train = int(0.8 * len(group))
        n_validation = int(0.1 * len(group))

        train.append(group.iloc[:n_train])
        validation.append(group.iloc[n_train:n_train + n_validation])
        test.append(group.iloc[n_train + n_validation:])

    return [pd.concat(p).sample(frac=1, random_state=SEED) for p in (train, validation, test)]


train_set, validation_set, test_set = stratified_split(dataset)

train_labels      = train_set["label"].map(label_to_number).to_numpy()
validation_labels = validation_set["label"].map(label_to_number).to_numpy()
test_labels       = test_set["label"].map(label_to_number).to_numpy()

for name, part in [("train", train_set), ("validation", validation_set), ("test", test_set)]:
    print(f"{name:<11} {len(part):>6}  ({part.groupby('label').size().unique()} per language)")

train        18400  ([800] per language)
validation    2300  ([100] per language)
test          2300  ([100] per language)


## 3. TF-IDF

In [ ]:
def get_ngrams(text):
    text = text.lower()
    words = text.split()
    grams = []

    for n in WORD_NGRAM_SIZES:
        grams += ["word:" + " ".join(words[i:i + n]) for i in range(len(words) - n + 1)]
    for n in CHAR_NGRAM_SIZES:
        grams += ["char:" + text[i:i + n] for i in range(len(text) - n + 1)]

    return grams


get_ngrams("big cat")

['word:big',
 'word:cat',
 'word:big cat',
 'char:bi',
 'char:ig',
 'char:g ',
 'char: c',
 'char:ca',
 'char:at',
 'char:big',
 'char:ig ',
 'char:g c',
 'char: ca',
 'char:cat',
 'char:big ',
 'char:ig c',
 'char:g ca',
 'char: cat']

In [ ]:
class TfidfVectorizer:

    def fit(self, documents):
        # how many documents contain each n-gram
        document_frequency = Counter()
        for document in documents:
            document_frequency.update(set(get_ngrams(document)))

        # keep the MAX_FEATURES most widespread n-grams, one column each
        top = document_frequency.most_common(MAX_FEATURES)
        self.vocabulary = {gram: column for column, (gram, _) in enumerate(top)}

        counts = np.array([df for _, df in top])
        self.idf = np.log((1 + len(documents)) / (1 + counts)) + 1
        return self

    def transform(self, documents):
        # one (row, column, value) triple per n-gram found; everything else stays zero
        rows, columns, values = [], [], []
        for row, document in enumerate(documents):
            for gram, count in Counter(get_ngrams(document)).items():
                if gram in self.vocabulary:             # unseen n-grams are ignored
                    rows.append(row)
                    columns.append(self.vocabulary[gram])
                    values.append(float(count))         # TF

        matrix = sparse.csr_matrix((values, (rows, columns)),
                                   shape=(len(documents), len(self.vocabulary)))

        matrix = matrix.multiply(self.idf).tocsr()      # TF -> TF-IDF

        lengths = np.sqrt(matrix.multiply(matrix).sum(axis=1))   # L2 length of each row
        lengths = np.asarray(lengths).ravel()
        lengths[lengths == 0] = 1
        return (sparse.diags(1 / lengths) @ matrix).tocsr()      # every row now has length 1

    def fit_transform(self, documents):
        return self.fit(documents).transform(documents)

In [ ]:
# fit on train only, so validation and test stay unseen
vectorizer = TfidfVectorizer()
train_matrix      = vectorizer.fit_transform(train_set["text"].tolist())
validation_matrix = vectorizer.transform(validation_set["text"].tolist())
test_matrix       = vectorizer.transform(test_set["text"].tolist())

print(train_matrix.shape, f"{train_matrix.nnz:,} non-zeros")

(18400, 50000) 2,908,969 non-zeros


## 4. Logistic regression (softmax)

`P = softmax(XW + b)`, cross-entropy loss, gradient `Xᵀ(P − Y) / n`, mini-batch gradient descent.

In [ ]:
def softmax(scores):
    scores = scores - scores.max(axis=1, keepdims=True)      # prevents exp() overflow
    exponentials = np.exp(scores)
    return exponentials / exponentials.sum(axis=1, keepdims=True)


class LogisticRegression:

    def fit(self, X, y, X_validation=None, y_validation=None):
        n_samples, n_features = X.shape
        n_classes = y.max() + 1

        self.W = np.zeros((n_features, n_classes))
        self.b = np.zeros(n_classes)
        rng = np.random.default_rng(SEED)

        for epoch in range(1, EPOCHS + 1):
            for batch in np.array_split(rng.permutation(n_samples), n_samples // BATCH_SIZE):
                X_batch = X[batch]

                error = softmax(X_batch @ self.W + self.b)
                error[np.arange(len(batch)), y[batch]] -= 1   # error = P - Y  (Y is one-hot)
                error /= len(batch)

                self.W -= LEARNING_RATE * (X_batch.T @ error)
                self.b -= LEARNING_RATE * error.sum(axis=0)

            if epoch % 5 == 0:
                probabilities = self.predict_probabilities(X)
                loss = -np.log(probabilities[np.arange(n_samples), y] + 1e-12).mean()
                message = f"epoch {epoch:>3}   loss {loss:.4f}"
                if X_validation is not None:
                    accuracy = (self.predict(X_validation) == y_validation).mean()
                    message += f"   validation acc {accuracy:.4f}"
                print(message)

        return self

    def predict_probabilities(self, X):
        return softmax(X @ self.W + self.b)

    def predict(self, X):
        return np.asarray(X @ self.W + self.b).argmax(axis=1)


classifier = LogisticRegression()
classifier.fit(train_matrix, train_labels, validation_matrix, validation_labels)

epoch   5   loss 0.3213   validation acc 0.9530


epoch  10   loss 0.2046   validation acc 0.9574


epoch  15   loss 0.1587   validation acc 0.9587


epoch  20   loss 0.1328   validation acc 0.9609


epoch  25   loss 0.1150   validation acc 0.9635


epoch  30   loss 0.1021   validation acc 0.9643


epoch  35   loss 0.0912   validation acc 0.9643


epoch  40   loss 0.0832   validation acc 0.9643


## 5. Macro-F1

In [ ]:
def confusion_matrix(true_labels, predicted_labels):
    counts = np.zeros((len(LABELS), len(LABELS)), dtype=int)
    for true, predicted in zip(true_labels, predicted_labels):
        counts[true][predicted] += 1                    # rows = true, columns = predicted
    return counts


def divide(a, b):
    return a / b if b else 0.0                          # 0/0 counts as 0, by convention


def precision_recall_f1(true_labels, predicted_labels):
    counts = confusion_matrix(true_labels, predicted_labels)
    precisions, recalls, f1_scores = [], [], []

    for k in range(len(LABELS)):
        true_positives  = counts[k][k]
        false_positives = counts[:, k].sum() - true_positives
        false_negatives = counts[k, :].sum() - true_positives

        precision = divide(true_positives, true_positives + false_positives)
        recall    = divide(true_positives, true_positives + false_negatives)
        f1        = divide(2 * precision * recall, precision + recall)

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

    return np.array(precisions), np.array(recalls), np.array(f1_scores), counts


def macro_f1(true_labels, predicted_labels):
    return precision_recall_f1(true_labels, predicted_labels)[2].mean()   # unweighted mean

## 6. Results

In [ ]:
for name, matrix, labels in [("validation", validation_matrix, validation_labels),
                             ("test",       test_matrix,       test_labels)]:
    predictions = classifier.predict(matrix)
    print(f"{name:<11} accuracy {(predictions == labels).mean():.4f}   "
          f"macro-F1 {macro_f1(labels, predictions):.4f}")

validation  accuracy 0.9643   macro-F1 0.9644
test        accuracy 0.9596   macro-F1 0.9592


In [ ]:
test_predictions = classifier.predict(test_matrix)
precisions, recalls, f1_scores, counts = precision_recall_f1(test_labels, test_predictions)

print(f"MACRO-F1 ON THE TEST SET = {f1_scores.mean():.4f}\n")

pd.DataFrame({
    "language":  [LANGUAGES[c] for c in LABELS],
    "precision": precisions.round(3),
    "recall":    recalls.round(3),
    "f1":        f1_scores.round(3),
    "support":   counts.sum(axis=1),
}, index=LABELS).sort_values("f1")

MACRO-F1 ON THE TEST SET = 0.9592



,language,precision,recall,f1,support
mr,Marathi,0.833,0.65,0.730,100
gom,Konkani,0.696,0.80,0.744,100
hi,Hindi,0.843,0.97,0.902,100
bd,Bodo,0.989,0.86,0.920,100
mai,Maithili,0.932,0.96,0.946,100
dg,Dogri,0.960,0.95,0.955,100
sa,Sanskrit,0.961,0.98,0.970,100
mni,Manipuri,1.000,0.95,0.974,100
ne,Nepali,0.971,0.99,0.980,100
as,Assamese,0.990,0.98,0.985,100


In [ ]:
pd.DataFrame(counts, index=LABELS, columns=LABELS)

,as,bd,bn,dg,gom,gu,hi,kha,kn,ks,...,mr,ne,or,pa,sa,sat,sd,ta,te,ur
as,98,0,2,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
bd,0,86,0,0,0,0,12,0,0,0,...,0,0,0,0,1,0,0,0,0,0
bn,0,0,100,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
dg,0,0,0,95,0,0,5,0,0,0,...,0,0,0,0,0,0,0,0,0,0
gom,0,0,0,1,80,0,1,0,0,0,...,13,2,0,0,1,0,0,0,0,0
gu,0,0,0,0,0,100,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
hi,0,1,0,0,0,0,97,0,0,0,...,0,0,0,0,0,0,0,0,0,0
kha,0,0,0,0,0,0,0,100,0,0,...,0,0,0,0,0,0,0,0,0,0
kn,0,0,0,0,0,0,0,0,100,0,...,0,0,0,0,0,0,0,0,0,0
ks,0,0,0,0,0,0,0,0,0,98,...,0,0,0,0,0,0,0,0,0,2


## 7. Try it

In [ ]:
def identify(sentences):
    probabilities = classifier.predict_probabilities(vectorizer.transform(sentences))
    for sentence, row in zip(sentences, probabilities):
        best = row.argsort()[::-1][:3]
        print(f"{sentence[:55]:<57} -> "
              + ",  ".join(f"{LANGUAGES[LABELS[i]]} {row[i]:.0%}" for i in best))


identify([
    "भारत एक विशाल और विविधतापूर्ण देश है जहाँ अनेक भाषाएँ बोली जाती हैं",
    "इंडिया हा एक मोठा देश आहे जिथे अनेक भाषा बोलल्या जातात",
    "இந்தியா பல மொழிகள் பேசப்படும் ஒரு பெரிய நாடு ஆகும்",
    "ভারত একটি বিশাল দেশ যেখানে বহু ভাষা প্রচলিত আছে",
    "ભારત એક વિશાળ દેશ છે જ્યાં અનેક ભાષાઓ બોલાય છે",
    "بھارت ایک بہت بڑا ملک ہے جہاں کئی زبانیں بولی جاتی ہیں",
])

भारत एक विशाल और विविधतापूर्ण देश है जहाँ अनेक भाषाएँ ब   -> Maithili 31%,  Hindi 30%,  Dogri 15%
इंडिया हा एक मोठा देश आहे जिथे अनेक भाषा बोलल्या जातात    -> Konkani 57%,  Marathi 35%,  Maithili 3%
இந்தியா பல மொழிகள் பேசப்படும் ஒரு பெரிய நாடு ஆகும்        -> Tamil 99%,  Konkani 0%,  Maithili 0%
ভারত একটি বিশাল দেশ যেখানে বহু ভাষা প্রচলিত আছে           -> Bengali 84%,  Assamese 14%,  Konkani 0%
ભારત એક વિશાળ દેશ છે જ્યાં અનેક ભાષાઓ બોલાય છે            -> Gujarati 98%,  Konkani 0%,  Maithili 0%
بھارت ایک بہت بڑا ملک ہے جہاں کئی زبانیں بولی جاتی ہیں    -> Urdu 99%,  Kashmiri 0%,  Sindhi 0%
